# ROGII Wellbore Geology — honest particle-filter model (no leak)

Uses only test-time data (horizontal MD/X/Y/Z/GR + typewell GR(TVT)). A
particle-filter geosteering tracker (momentum state, GR likelihood vs typewell,
seed ensemble weighted by likelihood) blended with a beam-search tracker and the
last-known TVT anchor. CV (per-well RMSE on train): ~8; pooled ~10.


In [ ]:
import os, glob, numpy as np, pandas as pd
def find_root():
    roots = ["/kaggle/input/rogii-wellbore-geology-prediction", "data"] + sorted(glob.glob("/kaggle/input/*"))
    for r in roots:
        if os.path.exists(os.path.join(r, "sample_submission.csv")) and glob.glob(os.path.join(r, "test", "*__horizontal_well.csv")):
            return r
    hits = glob.glob("/kaggle/input/**/*__horizontal_well.csv", recursive=True)
    if hits: return os.path.dirname(os.path.dirname(hits[0]))
    raise FileNotFoundError("data not found")
DATA = find_root(); print("DATA:", DATA)
sample = pd.read_csv(f"{DATA}/sample_submission.csv")
test_wells = sorted({os.path.basename(f).split("__")[0] for f in glob.glob(f"{DATA}/test/*__horizontal_well.csv")})
train_wells = {os.path.basename(f).split("__")[0] for f in glob.glob(f"{DATA}/train/*__horizontal_well.csv")}
print("test wells:", test_wells)

In [ ]:
"""
Particle-filter geosteering tracker (numpy, vectorised over particles).

State per particle: (pos, rate) where pos = TVT + Z  ("structural" position that
is ~constant when geology is flat), rate = d(pos)/dMD. At each eval step we
propagate with momentum, convert to TVT = pos - Z, score against the typewell
GR(TVT) curve via a Gaussian likelihood, resample, and take the weighted-mean TVT.

Predictions are made only for the eval rows (TVT_input is NaN). A seed ensemble
is combined with weights ∝ exp(loglik/scale). This is the core signal that gets
geosteering solutions to single-digit RMSE.
"""
import numpy as np

# tuned defaults (close to strong public solutions)
MOM = 0.998; VN = 0.002; PN = 0.005; RP = 0.1; RR = 0.001; RESAMP = 0.5


def _run_pf(md, z, gr, tw_tvt, tw_gr, last_tvt, last_Z, ir, gs,
            n_particles=500, seed=0):
    rng = np.random.default_rng(seed)
    N = n_particles
    ls = last_tvt + last_Z
    pos = ls + 2.0 * rng.standard_normal(N)
    rate = ir + 0.01 * rng.standard_normal(N)
    w = np.ones(N) / N
    lo, hi = tw_tvt[0] - 100, tw_tvt[-1] + 100
    out = np.empty(len(md)); loglik = 0.0
    prev_md = md[0] - 1.0
    for i in range(len(md)):
        dm = max(md[i] - prev_md, 1.0)
        rate = MOM * rate + VN * rng.standard_normal(N)
        pos = pos + rate * dm + PN * rng.standard_normal(N)
        tvt_p = np.clip(pos - z[i], lo, hi)
        pos = tvt_p + z[i]
        if np.isfinite(gr[i]):
            eg = np.interp(tvt_p, tw_tvt, tw_gr)
            d = (gr[i] - eg) / gs
            lk = np.maximum(np.exp(-0.5 * np.minimum(d * d, 600.0)), 1e-300)
            avg = float((w * lk).sum()); loglik += np.log(max(avg, 1e-300))
            w = w * lk; s = w.sum(); w = w / s if s > 0 else np.ones(N) / N
        neff = 1.0 / (w * w).sum()
        if neff < RESAMP * N:
            cum = np.cumsum(w); u0 = rng.uniform(0, 1.0 / N)
            idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)
            pos = pos[idx] + RP * rng.standard_normal(N)
            rate = rate[idx] + RR * rng.standard_normal(N)
            w = np.ones(N) / N
        out[i] = float(np.dot(w, pos - z[i]))
        prev_md = md[i]
    return out, loglik


def pf_predict(hw, tw, n_particles=500, n_seeds=48, scale=5.0):
    """Return a full-length TVT array (known rows kept, eval rows predicted)."""
    tw_s = tw.sort_values("TVT")
    tw_tvt = tw_s["TVT"].to_numpy(float)
    tw_gr = tw_s["GR"].fillna(tw_s["GR"].mean()).to_numpy(float)

    kn = hw[hw["TVT_input"].notna()]
    ev = hw[hw["TVT_input"].isna()]
    out = hw["TVT_input"].to_numpy(float).copy()
    if len(ev) == 0:
        return out

    last = kn.iloc[-1]
    last_tvt = float(last["TVT_input"]); last_Z = float(last["Z"])
    # GR noise sigma from known section
    tw_at_k = np.interp(kn["TVT_input"].to_numpy(), tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn["GR"].fillna(0).to_numpy() - tw_at_k), 10., 60.))
    # initial structural rate from last 30 known points
    tail = kn.tail(30)
    dt = np.diff(tail["TVT_input"].to_numpy()); dz = np.diff(tail["Z"].to_numpy())
    dmv = np.diff(tail["MD"].to_numpy()); m = dmv > 0
    ir = float(np.median((dt + dz)[m] / dmv[m])) if m.sum() >= 3 else 0.0

    gr_all = hw["GR"].interpolate(limit_direction="both").fillna(tw_gr.mean()).to_numpy(float)
    md = ev["MD"].to_numpy(float); z = ev["Z"].to_numpy(float)
    gr = gr_all[ev.index.to_numpy()]
    # prepend last known so momentum starts from the right place
    md = np.concatenate([[float(last["MD"])], md])
    z = np.concatenate([[last_Z], z])
    gr = np.concatenate([[np.nan], gr])

    preds = np.empty((n_seeds, len(md))); liks = np.empty(n_seeds)
    for s in range(n_seeds):
        preds[s], liks[s] = _run_pf(md, z, gr, tw_tvt, tw_gr, last_tvt, last_Z,
                                    ir, gs, n_particles, seed=s)
    liks -= liks.max()
    wts = np.exp(liks / scale); wts /= wts.sum()
    est = (wts[:, None] * preds).sum(0)[1:]        # drop the prepended known row
    out[ev.index.to_numpy()] = est
    return out


In [ ]:
"""Beam-search geosteering tracker (numba JIT). Complements the particle filter:
finds a low-cost monotone-ish path through the typewell GR(TVT) grid that matches
the horizontal GR signature. Ensembled over several stiffness configs."""
import numpy as np
from numba import njit
from scipy.signal import savgol_filter

# (beam_size, move_cost, err_scale, savgol_radius)
BEAM_CONFIGS = [
    (10, 20.0, 144.0, 2), (10, 8.0, 64.0, 2), (8, 35.0, 220.0, 1),
    (10, 14.0, 90.0, 5), (20, 4.0, 36.0, 3), (12, 12.0, 100.0, 3),
    (15, 25.0, 180.0, 2),
]


@njit(cache=True)
def _beam(sgr, tw_gr, si, BS, mc, es):
    n = len(sgr); nt = len(tw_gr); MAX = BS * 6
    bidx = np.zeros(BS, np.int64); bidx[0] = si
    bcost = np.full(BS, 1e30); bcost[0] = 0.0; bn = np.int64(1)
    hI = np.zeros((n, BS), np.int64); hP = np.zeros((n, BS), np.int64)
    cI = np.zeros(MAX, np.int64); cC = np.full(MAX, 1e30); cP = np.zeros(MAX, np.int64)
    for step in range(n):
        gv = sgr[step]; nc = np.int64(0)
        for bi in range(bn):
            idx = bidx[bi]; cost = bcost[bi]
            for d in range(-2, 3):
                ni = idx + d
                if ni < 0 or ni >= nt:
                    continue
                tot = cost + (gv - tw_gr[ni]) ** 2 / es + mc * (d if d >= 0 else -d)
                fnd = np.int64(-1)
                for ci in range(nc):
                    if cI[ci] == ni:
                        fnd = ci; break
                if fnd >= 0:
                    if tot < cC[fnd]:
                        cC[fnd] = tot; cP[fnd] = bi
                elif nc < MAX:
                    cI[nc] = ni; cC[nc] = tot; cP[nc] = bi; nc += 1
        kept = min(BS, nc)
        for i in range(kept):
            mi = i
            for j in range(i + 1, nc):
                if cC[j] < cC[mi]:
                    mi = j
            if mi != i:
                cI[i], cI[mi] = cI[mi], cI[i]
                cC[i], cC[mi] = cC[mi], cC[i]
                cP[i], cP[mi] = cP[mi], cP[i]
        hI[step, :kept] = cI[:kept]; hP[step, :kept] = cP[:kept]
        bidx[:kept] = cI[:kept]; bcost[:kept] = cC[:kept]; bn = kept
    best = np.int64(0)
    for b in range(1, bn):
        if bcost[b] < bcost[best]:
            best = b
    path = np.zeros(n, np.int64); b = best
    for s in range(n - 1, -1, -1):
        path[s] = hI[s, b]; b = hP[s, b]
    return path


def beam_predict(hw, tw):
    tw_s = tw.sort_values("TVT")
    tw_tvt = tw_s["TVT"].to_numpy(float)
    tw_gr = tw_s["GR"].fillna(tw_s["GR"].mean()).to_numpy(float)
    kn = hw[hw["TVT_input"].notna()]; ev = hw[hw["TVT_input"].isna()]
    out = hw["TVT_input"].to_numpy(float).copy()
    if len(ev) == 0:
        return out
    last_tvt = float(kn.iloc[-1]["TVT_input"])
    gr_all = hw["GR"].interpolate(limit_direction="both").fillna(tw_gr.mean()).to_numpy(float)
    hgr = gr_all[ev.index.to_numpy()]
    si = int(np.argmin(np.abs(tw_tvt - last_tvt)))
    paths = []
    for BS, mc, es, r in BEAM_CONFIGS:
        if r > 0 and len(hgr) > max(3, 2 * r + 1):
            win = min(2 * r + 1, len(hgr) if len(hgr) % 2 == 1 else len(hgr) - 1)
            sgr = savgol_filter(hgr, win, min(2, win - 1))
        else:
            sgr = hgr.copy()
        paths.append(tw_tvt[_beam(sgr.astype(np.float64), tw_gr, si, BS, float(mc), float(es))])
    out[ev.index.to_numpy()] = np.stack(paths, 0).mean(0)
    return out


In [ ]:
MODE_BLEND = (0.72, 0.18, 0.10)   # pf, beam, hold
parts = []
for w in test_wells:
    te = pd.read_csv(f"{DATA}/test/{w}__horizontal_well.csv")
    tw = pd.read_csv(f"{DATA}/test/{w}__typewell.csv")
    ev = te["TVT_input"].isna().to_numpy(); idx = np.where(ev)[0]
    pf = pf_predict(te, tw, n_particles=1000, n_seeds=160, scale=12.0)
    bm = beam_predict(te, tw)
    tps = float(te["TVT_input"].dropna().iloc[-1])
    a, b, c = MODE_BLEND
    pred = a * pf + b * bm + c * tps
    parts.append(pd.DataFrame({"id": [f"{w}_{i}" for i in idx], "tvt": pred[ev]}))

pred = pd.concat(parts, ignore_index=True)
sub = sample[["id"]].merge(pred, on="id", how="left")
sub["tvt"] = sub["tvt"].ffill().fillna(0.0)
sub.to_csv("submission.csv", index=False)
print("rows:", len(sub), "| nan:", sub["tvt"].isna().sum()); sub.head()